# RMIT Hackathon 2026 — Challenge 1
## Multilingual Prompt-Attack Classification (BENIGN / JAILBREAK / PROMPT_INJECTION)

**Chosen model: `FacebookAI/xlm-roberta-base`**
<!-- Đổi MODEL_KEY ở cell CONFIG và sửa dòng trên cho khớp. BTC yêu cầu ghi rõ model ở đầu notebook. -->

Pipeline: load data → class weights cho lớp hiếm → fine-tune encoder → đo macro-F1 trên validation →
tinh chỉnh trọng số lớp để tối đa macro-F1 → sinh `submission.csv`.

> ⚠️ Đặt tên notebook đúng mẫu `<university>_<campus>_<team>_C1`, ví dụ `RMIT_SGS_myteam_C1`.
> ⚠️ Chỉ có **6 lượt nộp** — luôn xem macro-F1 trên validation trước khi nộp.

In [ ]:
# === Cell 1: imports, compat helpers, GPU check ===
import os, glob, json, inspect, random
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)
from datasets import Dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); set_seed(SEED)

import transformers; print("transformers:", transformers.__version__)
print("GPUs:", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


def make_args(**kw):
    # TrainingArguments khác tên tham số giữa các bản transformers -> lọc cho khớp.
    sig = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in kw and "eval_strategy" not in sig:
        kw["evaluation_strategy"] = kw.pop("eval_strategy")
    dropped = [k for k in kw if k not in sig]
    if dropped:
        print("TrainingArguments bỏ qua:", dropped)
    return TrainingArguments(**{k: v for k, v in kw.items() if k in sig})


def tok_kwarg(tok):
    # Trainer mới dùng processing_class, bản cũ dùng tokenizer.
    sig = inspect.signature(Trainer.__init__).parameters
    return {"processing_class": tok} if "processing_class" in sig else {"tokenizer": tok}


def softmax(x):
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

In [ ]:
# === Cell 2: tìm đường dẫn dữ liệu + model (tự dò, không phụ thuộc cách attach input) ===
def find_data():
    for p in ["/kaggle/input/competitions/rmit-hackathon-2026",
              "/kaggle/input/rmit-hackathon-2026", "."]:
        if os.path.exists(os.path.join(p, "train.csv")):
            return p
    hits = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    raise FileNotFoundError("Không thấy train.csv — bấm 'Add Input' và thêm competition rmit-hackathon-2026.")


def find_model(slug):
    pats = [f"/kaggle/input/models/aisuko/{slug}/transformers/default/*/config.json",
            f"/kaggle/input/**/{slug}/**/config.json"]
    for pat in pats:
        hits = sorted(glob.glob(pat, recursive=True))
        if hits:
            return os.path.dirname(hits[0])
    hits = [h for h in glob.glob("/kaggle/input/**/config.json", recursive=True) if slug in h]
    if hits:
        return os.path.dirname(sorted(hits)[0])
    raise FileNotFoundError(f"Không thấy model '{slug}' — vào tab Models của cuộc thi, bấm 'Add Input'.")


DATA = find_data()
print("DATA =", DATA)

train_df = pd.read_csv(f"{DATA}/train.csv")
test_df  = pd.read_csv(f"{DATA}/test.csv")
sample   = pd.read_csv(f"{DATA}/sample_submission.csv")

LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
L2I = {l: i for i, l in enumerate(LABELS)}

print(train_df.shape, test_df.shape, sample.shape)
print("\nCột submission mẫu:", list(sample.columns))   # <- phải khớp đúng khi nộp
print("\nPhân bố nhãn:\n", train_df.label.value_counts(normalize=True).round(4))
print("\nPhân bố ngôn ngữ:\n", train_df.language.value_counts(normalize=True).round(4))
print("\nĐộ dài text (ký tự):\n", train_df.text.str.len().describe().round(1))
train_df.head(3)

In [ ]:
# === Cell 3 (TÙY CHỌN - CỨU HỘ): baseline TF-IDF ~2 phút, KHÔNG cần GPU ===
# Chỉ chạy nếu GPU chưa sẵn sàng và cần nộp gấp để chốt điểm Challenge 1.
# Bỏ qua cell này nếu transformer bên dưới chạy được bình thường.
RUN_TFIDF_FALLBACK = False

if RUN_TFIDF_FALLBACK:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.svm import LinearSVC

    tr_f, va_f = train_test_split(train_df, test_size=0.1, stratify=train_df.label, random_state=SEED)
    pipe = make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3,
                        sublinear_tf=True, max_features=300_000),
        LinearSVC(class_weight="balanced", C=1.0),
    )
    pipe.fit(tr_f.text, tr_f.label)
    print("TF-IDF val macro-F1 =", round(f1_score(va_f.label, pipe.predict(va_f.text), average="macro"), 4))

    sub_f = sample.copy()
    sub_f["label"] = pipe.fit(train_df.text, train_df.label).predict(test_df.text)
    sub_f.to_csv("submission_tfidf.csv", index=False)
    print(sub_f.label.value_counts())
    print("Đã lưu submission_tfidf.csv")

In [ ]:
# === Cell 4: CONFIG — chỉnh ở đây rồi Run All ===
MODELS = {
    # key                    : (slug trên Kaggle,                        lr,    epochs, bs/gpu, maxlen)
    "distilbert"             : ("distilbert-base-multilingual-cased",    3e-5,  3,      32,     256),
    "mmbert-small"           : ("mmbert-small",                          3e-5,  3,      32,     256),
    "mmbert-base"            : ("mmbert-base",                           2e-5,  3,      16,     256),
    "xlmr-base"              : ("xlm-roberta-base",                      2e-5,  5,      16,     256),
    "xlmr-large"             : ("xlm-roberta-large",                     1e-5,  3,      8,      256),
    "rembert"                : ("rembert",                               1e-5,  3,      8,      256),
    "deberta-pi"             : ("deberta-v3-base-prompt-injection-v2",   2e-5,  3,      16,     256),
    "wolf-defender"          : ("wolf-defender-prompt-injection",        2e-5,  3,      16,     256),
}

MODEL_KEY = "xlmr-base"        # <<< ĐỔI Ở ĐÂY (chạy lần 1 nên để "distilbert" cho nhanh)
SLUG, LR, EPOCHS, BS, MAXLEN = MODELS[MODEL_KEY]

USE_LANG_PREFIX = True         # thêm "[pt] " vào đầu text — tận dụng cột language
VAL_SIZE        = 0.10         # giữ cố định để các model ensemble được với nhau
GRAD_CKPT       = MODEL_KEY in ("xlmr-large",)   # tiết kiệm VRAM cho model lớn

MODEL_PATH = find_model(SLUG)
TAG = MODEL_KEY
print(f"MODEL_KEY={MODEL_KEY}\nMODEL_PATH={MODEL_PATH}\nlr={LR} epochs={EPOCHS} bs/gpu={BS} maxlen={MAXLEN}")

In [ ]:
# === Cell 5: tách train/val + tokenize ===
def prep(df):
    s = df.text.astype(str)
    if USE_LANG_PREFIX and "language" in df.columns:
        s = "[" + df.language.astype(str) + "] " + s
    return s.tolist()

tr, va = train_test_split(train_df, test_size=VAL_SIZE,
                          stratify=train_df.label, random_state=SEED)
print("train:", len(tr), "| val:", len(va), "| test:", len(test_df))

tok = AutoTokenizer.from_pretrained(MODEL_PATH)

def mk(df, labelled=True):
    d = {"text": prep(df)}
    if labelled:
        d["labels"] = df.label.map(L2I).tolist()
    ds = Dataset.from_dict(d)
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=MAXLEN),
                  batched=True, remove_columns=["text"])

ds_tr, ds_va, ds_te = mk(tr), mk(va), mk(test_df, labelled=False)
y_va = va.label.map(L2I).values
print(ds_tr)

In [ ]:
# === Cell 6: class weights + Trainer có trọng số (xử lý mất cân bằng -> then chốt cho macro-F1) ===
counts = tr.label.map(L2I).value_counts().sort_index().values
cw = torch.tensor(len(tr) / (len(LABELS) * counts), dtype=torch.float)
print("class weights:", dict(zip(LABELS, cw.numpy().round(3))))


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = F.cross_entropy(out.logits, labels, weight=cw.to(out.logits.device))
        return (loss, out) if return_outputs else loss


def compute_metrics(p):
    pred = p.predictions.argmax(-1)
    return {"f1_macro": f1_score(p.label_ids, pred, average="macro")}


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, num_labels=len(LABELS),
    id2label={i: l for l, i in L2I.items()}, label2id=L2I,
    ignore_mismatched_sizes=True,     # cần cho 2 model safety-tuned (head khác số lớp)
)
if GRAD_CKPT:
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

In [ ]:
# === Cell 7: train ===
args = make_args(
    output_dir=f"out-{TAG}",
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BS,
    per_device_eval_batch_size=BS * 2,
    warmup_ratio=0.06, warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=ds_tr, eval_dataset=ds_va,
    data_collator=DataCollatorWithPadding(tok),
    compute_metrics=compute_metrics,
    **tok_kwarg(tok),
)
trainer.train()

In [ ]:
# === Cell 8: đánh giá trên validation (macro-F1 + theo từng lớp + theo từng ngôn ngữ) ===
val_logits = trainer.predict(ds_va).predictions
val_probs  = softmax(val_logits)
val_pred   = val_probs.argmax(1)

base_f1 = f1_score(y_va, val_pred, average="macro")
print(f"VAL macro-F1 (argmax) = {base_f1:.4f}\n")
print(classification_report(y_va, val_pred, target_names=LABELS, digits=4))
print("Confusion matrix (hàng = thật, cột = đoán):")
print(pd.DataFrame(confusion_matrix(y_va, val_pred), index=LABELS, columns=LABELS))

print("\nmacro-F1 theo ngôn ngữ:")
for lg in sorted(va.language.unique()):
    m = (va.language == lg).values
    print(f"  {lg}: {f1_score(y_va[m], val_pred[m], average='macro'):.4f}  (n={m.sum()})")

In [ ]:
# === Cell 9: tinh chỉnh trọng số lớp để tối đa macro-F1 (thường +0.01~0.04, miễn phí) ===
def tune_weights(probs, y, rounds=4, grid=np.arange(0.4, 3.05, 0.05)):
    w = np.ones(probs.shape[1])
    best = f1_score(y, (probs * w).argmax(1), average="macro")
    for _ in range(rounds):
        improved = False
        for c in range(probs.shape[1]):
            for g in grid:
                w2 = w.copy(); w2[c] = g
                s = f1_score(y, (probs * w2).argmax(1), average="macro")
                if s > best + 1e-6:
                    best, w, improved = s, w2, True
        if not improved:
            break
    return w, best


W, tuned_f1 = tune_weights(val_probs, y_va)
print("trọng số lớp:", dict(zip(LABELS, W.round(3))))
print(f"VAL macro-F1: {base_f1:.4f} -> {tuned_f1:.4f}  (chênh {tuned_f1 - base_f1:+.4f})")

USE_TUNED = tuned_f1 > base_f1 + 0.002      # chỉ dùng nếu cải thiện rõ, tránh overfit val
print("Dùng trọng số đã tinh chỉnh:", USE_TUNED)
print("\n", classification_report(y_va, (val_probs * (W if USE_TUNED else 1)).argmax(1),
                                  target_names=LABELS, digits=4))

In [ ]:
# === Cell 10: dự đoán test + sinh submission.csv + lưu probs để ensemble ===
test_probs = softmax(trainer.predict(ds_te).predictions)
final_pred = (test_probs * (W if USE_TUNED else 1)).argmax(1)

sub = sample.copy()
id_col, lab_col = sub.columns[0], sub.columns[1]      # bám đúng tên cột của file mẫu
sub[lab_col] = [LABELS[i] for i in final_pred]

# --- kiểm tra trước khi nộp ---
assert len(sub) == 6000, f"Phải có 6000 dòng, đang có {len(sub)}"
assert set(sub[lab_col]) <= set(LABELS), f"Nhãn lạ: {set(sub[lab_col]) - set(LABELS)}"
assert list(sub[id_col]) == list(test_df[test_df.columns[0]]), "id không khớp test.csv"
assert sub[lab_col].isna().sum() == 0

sub.to_csv("submission.csv", index=False)
np.save(f"testprobs_{TAG}.npy", test_probs)
np.save(f"valprobs_{TAG}.npy", val_probs)
np.save("val_y.npy", y_va)
va[[va.columns[0]]].to_csv("val_ids.csv", index=False)

print(sub[lab_col].value_counts(normalize=True).round(4))
print(f"\nVAL macro-F1 = {max(base_f1, tuned_f1 if USE_TUNED else base_f1):.4f}  | model = {SLUG}")
print("Đã lưu submission.csv")
sub.head()

---
## Bước tiếp theo

**Nộp bài:** tab *Output* → tải `submission.csv` → vào trang cuộc thi → **Submit Predictions** → upload.
Hoặc **Save Version → Save & Run All**, rồi submit từ output của notebook.

**Ensemble (sau khi chạy ≥2 model khác nhau):**
1. Mỗi lần chạy xong một model, **Save Version** để output được giữ lại.
2. Trong notebook mới: **Add Input → Your Work → Notebook** → chọn các lần chạy trước.
3. Chạy cell ensemble bên dưới.

**Thứ tự ưu tiên khi còn ít thời gian**
| Lượt | Model | Thời gian ước tính |
|---|---|---|
| 1 | `distilbert` (chốt điểm Challenge 1) | ~5 phút |
| 2 | `xlmr-base` | ~15 phút |
| 3 | `deberta-pi` hoặc `wolf-defender` (đã biết prompt injection) | ~15 phút |
| 4 | `xlmr-large` (mạnh nhất, cần VRAM) | ~40 phút |
| 5–6 | ensemble | ~5 phút |

In [ ]:
# === Cell 11 (TÙY CHỌN): ensemble nhiều model đã chạy ===
# Cần: Add Input -> Your Work -> Notebook (chọn các lần chạy trước có testprobs_*.npy)
RUN_ENSEMBLE = False

if RUN_ENSEMBLE:
    vp = sorted(glob.glob("/kaggle/input/**/valprobs_*.npy", recursive=True))
    tp = sorted(glob.glob("/kaggle/input/**/testprobs_*.npy", recursive=True))
    print("val probs:", [os.path.basename(p) for p in vp])
    print("test probs:", [os.path.basename(p) for p in tp])
    assert len(vp) == len(tp) and len(vp) >= 2, "Cần ít nhất 2 model, và val/test phải khớp nhau"

    y_ref = np.load(sorted(glob.glob("/kaggle/input/**/val_y.npy", recursive=True))[0])
    V = np.mean([np.load(p) for p in vp], axis=0)
    T = np.mean([np.load(p) for p in tp], axis=0)

    for p in vp:
        print(f"  {os.path.basename(p):28s} {f1_score(y_ref, np.load(p).argmax(1), average='macro'):.4f}")
    print(f"  {'ENSEMBLE (trung bình)':28s} {f1_score(y_ref, V.argmax(1), average='macro'):.4f}")

    We, fe = tune_weights(V, y_ref)
    print(f"  {'ENSEMBLE + tinh chỉnh':28s} {fe:.4f}   trọng số={We.round(3)}")

    sub_e = sample.copy()
    sub_e[sub_e.columns[1]] = [LABELS[i] for i in (T * We).argmax(1)]
    assert len(sub_e) == 6000 and set(sub_e[sub_e.columns[1]]) <= set(LABELS)
    sub_e.to_csv("submission.csv", index=False)
    print("\nĐã ghi đè submission.csv bằng bản ensemble")
    print(sub_e[sub_e.columns[1]].value_counts(normalize=True).round(4))